# Phase 3 — Feature Engineering & Preprocessing (Days 23–47)
### CampusX: 100 Days of Machine Learning

> **Goal of this phase:** Transform raw, messy data into a clean, model-ready feature matrix. This is the most important phase — feature engineering contributes more to model performance than algorithm choice.

---

## Day 23 — Introduction to Feature Engineering

### What is Feature Engineering?
Feature engineering is the process of using **domain knowledge and data transformations** to create, modify, or select features (input variables) that make ML models more accurate.

> *"Coming up with features is difficult, time-consuming, requires expert knowledge. Applied machine learning is basically feature engineering." — Andrew Ng*

### Why Raw Data is Never Model-Ready
1. **Missing values** — most algorithms cannot handle NaN
2. **Categorical strings** — algorithms need numbers
3. **Different scales** — feature with range [0, 1000] dominates one with [0, 1]
4. **Skewed distributions** — violates normality assumptions of linear models
5. **Outliers** — distort model parameters
6. **Irrelevant features** — add noise and increase overfitting
7. **Non-linear relationships** — may need transformation to become linear

### The Feature Engineering Pipeline (Overview)
```
Raw Data
   ↓ Handle Missing Values
   ↓ Handle Outliers
   ↓ Encode Categorical Features
   ↓ Scale Numeric Features
   ↓ Transform Distributions
   ↓ Create New Features
   ↓ Select Best Features
   ↓ Model-Ready Feature Matrix X
```

### Types of Feature Engineering

| Type | Description | Example |
|---|---|---|
| **Feature scaling** | Bring features to same scale | Standardisation, Normalisation |
| **Feature encoding** | Convert categories to numbers | One-hot, ordinal, target encoding |
| **Feature transformation** | Change distribution shape | Log, Box-Cox, sqrt |
| **Feature construction** | Create new features | BMI = weight/height², age_groups |
| **Feature splitting** | Extract information from compound columns | Split "New York, USA" → city, country |
| **Date/time features** | Extract temporal info | Extract month, weekday, is_holiday |
| **Text features** | Convert text to numbers | TF-IDF, word embeddings |
| **Missing value handling** | Fill or flag NaN values | Mean imputation, KNN imputer |
| **Outlier handling** | Remove or cap extremes | IQR capping, Z-score removal |
| **Feature selection** | Remove irrelevant/redundant features | RFE, SelectKBest, Lasso |

---

## Day 24 — Standardisation (StandardScaler)

### Why Scale Features?
Many ML algorithms are **scale-sensitive** — features measured in large units (income in thousands) will dominate features measured in small units (age 0–100) when computing distances or gradients.

**Algorithms sensitive to scale:**
- K-Nearest Neighbours (uses distance)
- SVM (uses distances in feature space)
- PCA (maximises variance — large-scale features dominate)
- Neural Networks (gradient descent converges poorly with unscaled features)
- Logistic/Linear Regression with regularisation (penalty applies equally to all weights)

**Algorithms NOT sensitive to scale:**
- Decision Trees (use splits, not distances)
- Random Forest
- Gradient Boosting (XGBoost, LightGBM)
- Naive Bayes

### What is Standardisation?
Standardisation (Z-score normalisation) transforms each feature to have **mean = 0** and **standard deviation = 1**.

**Formula:**
```
z = (x - μ) / σ
```
Where:
- x = original value
- μ = mean of the feature
- σ = standard deviation of the feature

### Implementation

```python
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

df = pd.read_csv('winequality-red.csv')
X = df.drop('quality', axis=1)
y = df['quality']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# CRITICAL: Fit ONLY on training data, transform both train and test
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit + transform
X_test_scaled  = scaler.transform(X_test)       # transform only (no fit!)

# The scaler learns: mean and std of each column from training data
print("Means learned:", scaler.mean_)
print("Std devs learned:", scaler.scale_)

# Verify
X_train_df = pd.DataFrame(X_train_scaled, columns=X.columns)
print(X_train_df.describe().round(2))
# mean ≈ 0, std ≈ 1 for every column
```

### Why Fit Only on Training Data?
Fitting the scaler on the full dataset (including test) constitutes **data leakage** — the test set statistics influence preprocessing, making performance estimates overly optimistic. In production, you'll only have training data statistics available.

### Before vs After Visualisation

```python
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
cols = ['fixed acidity', 'volatile acidity', 'alcohol']

for i, col in enumerate(cols):
    axes[0, i].hist(X_train[col], bins=30, color='steelblue', alpha=0.7)
    axes[0, i].set_title(f'Before: {col}')

    axes[1, i].hist(X_train_scaled[:, X.columns.get_loc(col)], bins=30,
                    color='coral', alpha=0.7)
    axes[1, i].set_title(f'After StandardScaler: {col}')

plt.suptitle('Effect of StandardScaler', fontsize=14)
plt.tight_layout()
plt.show()
```

---

## Day 25 — Normalisation (MinMaxScaler)

### What is Normalisation?
Normalisation (Min-Max Scaling) maps each feature to a fixed range — typically **[0, 1]** or [-1, 1].

**Formula:**
```
x_scaled = (x - x_min) / (x_max - x_min)
```

### When to Use Normalisation vs Standardisation

| Situation | Use |
|---|---|
| Data is normally distributed | StandardScaler |
| Data has outliers | StandardScaler (more robust) |
| Need values in [0,1] range | MinMaxScaler |
| Neural network inputs | MinMaxScaler or StandardScaler |
| Image pixel values | MinMaxScaler (divide by 255) |
| Algorithm assumes bounded input | MinMaxScaler |

**Key difference:** StandardScaler preserves outlier effects (they become extreme z-scores). MinMaxScaler is sensitive to outliers — one extreme value compresses all others into a tiny range.

### Implementation

```python
from sklearn.preprocessing import MinMaxScaler, RobustScaler, MaxAbsScaler

# MinMaxScaler — maps to [0, 1]
mms = MinMaxScaler()
X_train_mms = mms.fit_transform(X_train)
X_test_mms  = mms.transform(X_test)

# Custom range [−1, 1]
mms_custom = MinMaxScaler(feature_range=(-1, 1))
X_train_custom = mms_custom.fit_transform(X_train)

# RobustScaler — uses median and IQR instead of mean/std
# Much less sensitive to outliers
from sklearn.preprocessing import RobustScaler
rs = RobustScaler()
X_train_robust = rs.fit_transform(X_train)
# Formula: (x - Q2) / (Q3 - Q1)

# Verify
print(X_train_mms.min(), X_train_mms.max())  # 0.0, 1.0
```

### Comparing All Scalers

```python
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
import matplotlib.pyplot as plt
import numpy as np

# Data with outlier
data = np.array([10, 20, 15, 25, 14, 18, 1000]).reshape(-1, 1)  # 1000 is an outlier

scalers = {
    'StandardScaler': StandardScaler(),
    'MinMaxScaler': MinMaxScaler(),
    'RobustScaler': RobustScaler()
}

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].scatter(range(len(data)), data, color='gray')
axes[0].set_title('Original (with outlier)')

for i, (name, scaler) in enumerate(scalers.items(), 1):
    scaled = scaler.fit_transform(data)
    axes[i].scatter(range(len(scaled)), scaled, color='steelblue')
    axes[i].set_title(name)

plt.tight_layout()
plt.show()
```

---

## Day 26 — Ordinal & Label Encoding

### Label Encoding
Label Encoding converts each category to an integer. Sklearn's `LabelEncoder` assigns 0, 1, 2, ... in alphabetical order.

```python
from sklearn.preprocessing import LabelEncoder
import pandas as pd

# Target variable encoding (y)
le = LabelEncoder()
y_encoded = le.fit_transform(['cat', 'dog', 'fish', 'cat', 'dog'])
# [0, 1, 2, 0, 1]

print(le.classes_)          # ['cat', 'dog', 'fish']
print(le.inverse_transform([0, 2]))  # ['cat', 'fish']
```

**WARNING:** Never use LabelEncoder on input features that are nominal (no order). Encoding `['Red', 'Green', 'Blue']` as `[2, 1, 0]` implies Red > Green > Blue — a false relationship that misleads the model.

**Use LabelEncoder only for:** the target variable (y) in classification problems.

### Ordinal Encoding
For categorical features that have a **natural, meaningful order**.

```python
from sklearn.preprocessing import OrdinalEncoder
import pandas as pd

df = pd.DataFrame({
    'Size': ['S', 'M', 'L', 'XL', 'M', 'S'],
    'Quality': ['Poor', 'Average', 'Good', 'Excellent', 'Good', 'Poor'],
    'Price': [100, 200, 300, 400, 250, 120]
})

# Define the order for each feature
oe = OrdinalEncoder(categories=[
    ['S', 'M', 'L', 'XL'],                        # Size
    ['Poor', 'Average', 'Good', 'Excellent']        # Quality
])

df[['Size_enc', 'Quality_enc']] = oe.fit_transform(df[['Size', 'Quality']])
print(df)
# S→0, M→1, L→2, XL→3 | Poor→0, Average→1, Good→2, Excellent→3
```

### Handling Unknown Categories at Test Time

```python
# Handle new categories not seen during training
oe = OrdinalEncoder(
    categories=[['S', 'M', 'L', 'XL']],
    handle_unknown='use_encoded_value',
    unknown_value=-1  # assign -1 to unknown categories
)
```

---

## Day 27 — One Hot Encoding

### What is One-Hot Encoding?
One-Hot Encoding (OHE) converts a categorical column with N unique values into N binary (0/1) columns. Each column represents one category — 1 if the sample belongs to that category, 0 otherwise.

**When to use:** Nominal categorical features with no ordering (country, colour, product type).

### Example
```
Original: Color = ['Red', 'Green', 'Blue', 'Red']

After OHE:
Color_Red  Color_Green  Color_Blue
    1           0            0
    0           1            0
    0           0            1
    1           0            0
```

### Implementation with Pandas

```python
import pandas as pd

df = pd.DataFrame({'Color': ['Red', 'Green', 'Blue', 'Red', 'Green']})

# pd.get_dummies — quick and easy
df_encoded = pd.get_dummies(df, columns=['Color'], drop_first=True)
# drop_first=True drops one column to avoid dummy variable trap
print(df_encoded)
```

### Implementation with Sklearn (Preferred for Pipelines)

```python
from sklearn.preprocessing import OneHotEncoder
import numpy as np

ohe = OneHotEncoder(
    drop='first',           # drop first category (avoid dummy trap)
    sparse_output=False,    # return dense array
    handle_unknown='ignore' # ignore categories not seen during training
)

X_cat = df[['Color']].values
ohe.fit(X_cat)
X_encoded = ohe.transform(X_cat)
print(ohe.categories_)     # [array(['Blue', 'Green', 'Red'])]
print(ohe.get_feature_names_out())  # ['Color_Green', 'Color_Red']
```

### The Dummy Variable Trap
If you one-hot encode a 3-category feature without dropping one column, you create **perfect multicollinearity** — one column is perfectly predictable from the others (Blue = 1 - Red - Green). This makes linear models unstable. Always use `drop='first'` or `drop_first=True`.

### High Cardinality Problem
If a feature has 1000 unique values (e.g., city name), OHE creates 1000 columns — this explodes your feature space and causes the curse of dimensionality. Solutions:
- **Target Encoding** — replace category with mean target value (use carefully, causes leakage)
- **Frequency Encoding** — replace with count/frequency of each category
- **Embedding** — use neural network embeddings (for very high cardinality)
- **Group rare categories** into an "Other" bucket

```python
# Group rare categories
threshold = 50  # categories appearing < 50 times → 'Other'
counts = df['City'].value_counts()
df['City'] = df['City'].where(df['City'].isin(counts[counts >= threshold].index), 'Other')
```

---

## Day 28 — Column Transformer

### The Problem Without ColumnTransformer
A real dataset has both numeric and categorical columns, each needing different preprocessing. Without ColumnTransformer, you'd manually split, preprocess, and recombine — messy and error-prone.

### What is ColumnTransformer?
`ColumnTransformer` applies **different transformers to different columns** and concatenates the results into a single feature matrix. This is the building block of production ML pipelines.

### Implementation

```python
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

df = pd.read_csv('titanic.csv')
X = df[['Age', 'Fare', 'Pclass', 'Sex', 'Embarked']]
y = df['Survived']

# Define column groups
numeric_features    = ['Age', 'Fare']
categorical_features = ['Sex', 'Embarked']
passthrough_features = ['Pclass']  # already numeric, no scaling needed

# Define preprocessing for each group
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first'))
])

# Combine into ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num',  numeric_transformer,      numeric_features),
        ('cat',  categorical_transformer,  categorical_features),
        ('pass', 'passthrough',            passthrough_features)
    ],
    remainder='drop'  # drop all other columns not listed
)

# Fit and transform
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed  = preprocessor.transform(X_test)

print("Processed shape:", X_train_processed.shape)
print("Feature names:", preprocessor.get_feature_names_out())
```

---

## Day 29 — Sklearn Pipelines

### What is a Pipeline?
A `Pipeline` chains multiple preprocessing steps and a final estimator into a **single object** that can be trained, evaluated, and serialised as a unit.

### Why Pipelines are Critical
1. **Prevents data leakage** — preprocessing is fit inside cross-validation folds automatically
2. **Reproducibility** — one object encodes the entire transformation
3. **Deployment simplicity** — one object to serialise and load
4. **Code cleanliness** — no intermediate variables, no manual fit/transform calls

### Full Pipeline Example

```python
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, train_test_split
import pandas as pd

df = pd.read_csv('titanic.csv')
X = df[['Age', 'Fare', 'Pclass', 'Sex', 'Embarked']]
y = df['Survived']

numeric_features     = ['Age', 'Fare', 'Pclass']
categorical_features = ['Sex', 'Embarked']

preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), numeric_features),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore', drop='first'))
    ]), categorical_features)
])

# Full end-to-end pipeline
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

# Evaluate with cross-validation (leakage-free)
scores = cross_val_score(full_pipeline, X, y, cv=5, scoring='accuracy')
print(f"CV Accuracy: {scores.mean():.4f} ± {scores.std():.4f}")

# Train on full training set and evaluate on test set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
full_pipeline.fit(X_train, y_train)
print(f"Test Accuracy: {full_pipeline.score(X_test, y_test):.4f}")

# Save the entire pipeline (preprocessing + model)
import joblib
joblib.dump(full_pipeline, 'titanic_pipeline.pkl')

# Load and predict on new data
pipeline = joblib.load('titanic_pipeline.pkl')
new_passenger = pd.DataFrame({'Age': [28], 'Fare': [50.0], 'Pclass': [2],
                               'Sex': ['female'], 'Embarked': ['S']})
print(pipeline.predict_proba(new_passenger))
```

---

## Day 30 — Function Transformer (Log, Reciprocal, Sqrt)

### Why Transform Distributions?
Many ML algorithms (linear models, SVM) assume features are **approximately normally distributed**. Right-skewed features (income, house prices, website traffic) violate this assumption and can degrade model performance.

**Mathematical transforms** can reduce skewness and make distributions more symmetric.

### Log Transform

**Use when:** Feature is right-skewed and all values are positive.

```python
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline

df = pd.read_csv('train.csv')  # House prices

# Manual log transform
df['SalePrice_log'] = np.log(df['SalePrice'])      # natural log
df['SalePrice_log1p'] = np.log1p(df['SalePrice'])  # log(1+x), safe for 0 values

# Sklearn FunctionTransformer (for use in pipelines)
log_transformer = FunctionTransformer(np.log1p, validate=True)
log_transformer.fit(df[['LotArea']])
df['LotArea_log'] = log_transformer.transform(df[['LotArea']])

# Visualise
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
df['SalePrice'].hist(bins=50, ax=axes[0,0])
axes[0,0].set_title('SalePrice — Original (right-skewed)')
df['SalePrice_log'].hist(bins=50, ax=axes[0,1])
axes[0,1].set_title('SalePrice — After log (near-normal)')
df['LotArea'].hist(bins=50, ax=axes[1,0])
axes[1,0].set_title('LotArea — Original')
df['LotArea_log'].hist(bins=50, ax=axes[1,1])
axes[1,1].set_title('LotArea — After log')
plt.tight_layout()
plt.show()

print("Skewness before log:", df['SalePrice'].skew().round(2))
print("Skewness after log: ", df['SalePrice_log'].skew().round(2))
```

### Square Root Transform

**Use when:** Mild right skew. Also useful for count data (Poisson distributed).

```python
df['SalePrice_sqrt'] = np.sqrt(df['SalePrice'])
print("Skewness after sqrt:", df['SalePrice_sqrt'].skew().round(2))
```

### Reciprocal Transform

**Use when:** Strong right skew, very large range differences. Values must be non-zero.

```python
df['col_reciprocal'] = 1 / df['some_col']
```

### When to Use Which

| Skewness | Transform |
|---|---|
| Mild right skew (1–2) | Square root |
| Moderate right skew (2–5) | Log |
| Severe right skew (>5) | Log or Box-Cox |
| Left skew | Square, exponential |
| Negative values | Yeo-Johnson |

---

## Day 31 — Power Transformer (Box-Cox & Yeo-Johnson)

### Box-Cox Transform
The Box-Cox transform is a **family of power transforms** parametrised by λ that automatically finds the best λ to make the distribution as normal as possible.

```
x_transformed = (x^λ - 1) / λ   (if λ ≠ 0)
x_transformed = log(x)           (if λ = 0)
```

Special cases: λ=1 (identity), λ=0 (log), λ=0.5 (sqrt), λ=-1 (reciprocal)

**Constraint:** All values must be **strictly positive** (x > 0).

### Yeo-Johnson Transform
Extends Box-Cox to **handle zero and negative values**. Preferred in most practical situations.

### Implementation

```python
from sklearn.preprocessing import PowerTransformer
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

df = pd.read_csv('train.csv')

# Select right-skewed numeric columns
skewed_cols = df.select_dtypes(include=np.number).apply(lambda x: x.skew()).sort_values(ascending=False)
skewed_cols = skewed_cols[skewed_cols > 0.75].index.tolist()
print(f"Skewed columns: {skewed_cols}")

# Box-Cox (positive values only)
pt_bc = PowerTransformer(method='box-cox', standardize=True)
# Yeo-Johnson (any values)
pt_yj = PowerTransformer(method='yeo-johnson', standardize=True)

# Apply Yeo-Johnson to all skewed columns
df_skewed = df[skewed_cols].fillna(df[skewed_cols].median())
df_transformed = pt_yj.fit_transform(df_skewed)
df_transformed = pd.DataFrame(df_transformed, columns=skewed_cols)

print("Lambdas learned:", dict(zip(skewed_cols, pt_yj.lambdas_)))

# Compare skewness
original_skew   = df[skewed_cols].skew()
transformed_skew = df_transformed.skew()
comparison = pd.DataFrame({'Original': original_skew, 'Transformed': transformed_skew})
print(comparison)
```

---

## Day 32 — Binning & Binarization

### What is Binning (Discretisation)?
Converting a continuous feature into discrete bins (intervals). This can help tree-based models and reveal non-linear patterns.

### Equal-Width Binning
Each bin has the same range width.

```python
import pandas as pd
import numpy as np

df = pd.read_csv('diabetes.csv')

# Equal-width bins
df['Age_binned'] = pd.cut(df['Age'], bins=4, labels=['Young', 'Middle', 'Senior', 'Elderly'])

# Custom bin edges
df['BMI_binned'] = pd.cut(df['BMI'],
                           bins=[0, 18.5, 25, 30, 100],
                           labels=['Underweight', 'Normal', 'Overweight', 'Obese'])
print(df['BMI_binned'].value_counts())
```

### Equal-Frequency (Quantile) Binning
Each bin has the same number of samples.

```python
# Equal-frequency bins (quantile-based)
df['Age_qbinned'] = pd.qcut(df['Age'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])

# Using Sklearn KBinsDiscretizer
from sklearn.preprocessing import KBinsDiscretizer

kbd = KBinsDiscretizer(n_bins=4, encode='ordinal', strategy='quantile')
df['Age_kbins'] = kbd.fit_transform(df[['Age']])
print(kbd.bin_edges_)
```

### Binarization
Convert a numeric feature to binary (0 or 1) based on a threshold.

```python
from sklearn.preprocessing import Binarizer

# Glucose > 125 → diabetic risk flag (1), else 0
binarizer = Binarizer(threshold=125)
df['High_Glucose'] = binarizer.transform(df[['Glucose']])
```

### When Binning Helps
- **Non-monotonic relationships** — binning can capture U-shapes or inverted U-shapes that linear models miss
- **Reducing noise** — smooths out measurement noise in continuous features
- **Domain knowledge** — BMI categories have clinical meaning
- **Missing pattern** — a special bin for imputed values

### When Binning Hurts
- **Loss of information** — you discard the exact value
- **Arbitrary thresholds** — different binning choices give different results
- **For tree models** — not needed; they automatically find splits

---

## Day 33 — Handling Mixed Variables

### What are Mixed Variables?
Columns that contain **both numeric and categorical information** within a single cell. Common in real-world datasets from non-standardised data entry.

**Examples:**
- `"100 days"`, `"N/A"`, `"35 years old"` (numeric + unit)
- `"Yes"`, `"No"`, `"Maybe"`, `"5"` (categorical + numeric mixed)
- `"£50,000"` (currency symbol embedded in number)
- `"1st"`, `"2nd"`, `"3rd"` (rank strings)

### Strategies for Handling Mixed Variables

#### Strategy 1: Extract Numeric Component
```python
import pandas as pd
import re

df = pd.DataFrame({'Duration': ['5 days', '12 days', '3 days', 'Unknown', '7 days']})

# Extract number using regex
df['Duration_num'] = df['Duration'].str.extract(r'(\d+)').astype(float)

# Flag the non-numeric rows
df['Duration_missing'] = df['Duration'].str.contains(r'\d+', regex=True) == False
print(df)
```

#### Strategy 2: Separate into Two Columns
```python
df = pd.DataFrame({
    'Experience': ['3 years', '5 years', 'Fresher', '10 years', 'N/A']
})

# Numeric part
df['Exp_years'] = df['Experience'].str.extract(r'(\d+)').astype(float)

# Categorical part
df['Exp_category'] = df['Experience'].apply(
    lambda x: 'Fresher' if x == 'Fresher' else ('Missing' if x == 'N/A' else 'Experienced')
)
```

#### Strategy 3: Target Encoding for Mixed Columns
When the mixed column has too many unique formats, treat the entire column as categorical and apply target encoding.

---

## Day 34 — Date & Time Features

### Why Date/Time Engineering Matters
Raw datetime objects are not directly usable by ML models. But they contain rich predictive signals that need to be extracted manually.

### Basic Extraction

```python
import pandas as pd

df = pd.read_csv('bike_sharing.csv', parse_dates=['datetime'])
df['datetime'] = pd.to_datetime(df['datetime'])

# Extract all standard components
df['year']       = df['datetime'].dt.year
df['month']      = df['datetime'].dt.month          # 1-12
df['day']        = df['datetime'].dt.day            # 1-31
df['hour']       = df['datetime'].dt.hour           # 0-23
df['minute']     = df['datetime'].dt.minute
df['dayofweek']  = df['datetime'].dt.dayofweek     # 0=Monday, 6=Sunday
df['dayofyear']  = df['datetime'].dt.dayofyear     # 1-366
df['weekofyear'] = df['datetime'].dt.isocalendar().week
df['quarter']    = df['datetime'].dt.quarter       # 1-4
df['is_weekend'] = df['dayofweek'].isin([5, 6]).astype(int)
df['is_month_start'] = df['datetime'].dt.is_month_start.astype(int)
df['is_month_end']   = df['datetime'].dt.is_month_end.astype(int)
```

### Cyclic Encoding for Periodic Features
Hour, month, and day-of-week are **cyclic** — hour 23 is close to hour 0. Simple numeric encoding fails to capture this. Use sine/cosine encoding:

```python
import numpy as np

# Hour (24-hour cycle)
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

# Month (12-month cycle)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

# Day of week (7-day cycle)
df['dow_sin'] = np.sin(2 * np.pi * df['dayofweek'] / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['dayofweek'] / 7)
```

### Elapsed Time Features

```python
# Days since a reference date
reference_date = pd.Timestamp('2020-01-01')
df['days_since_ref'] = (df['datetime'] - reference_date).dt.days

# Time since last event (lag feature for time series)
df = df.sort_values('datetime')
df['hours_since_prev'] = df['datetime'].diff().dt.total_seconds() / 3600
```

### Holiday Features

```python
# pip install holidays
import holidays

in_holidays = holidays.India(years=df['year'].unique().tolist())
df['is_holiday'] = df['datetime'].dt.date.apply(lambda d: int(d in in_holidays))
```

---

## Days 35–40 — Handling Missing Values

### Understanding Missing Data Mechanisms

Before choosing an imputation strategy, understand WHY data is missing:

**MCAR — Missing Completely At Random:** Missingness is unrelated to any variable. Safe to drop rows. Example: a sensor randomly failed.

**MAR — Missing At Random:** Missingness is related to other observed variables but not the missing value itself. Example: older patients less likely to report income, but income itself doesn't affect reporting.

**MNAR — Missing Not At Random:** Missingness is related to the value itself. Example: people with very high income refuse to report it. Most problematic — dropping rows causes bias.

### Day 35: Complete Case Analysis (Listwise Deletion)

```python
# Simply drop rows with any missing values
df_clean = df.dropna()

# Drop rows missing in specific columns only
df_clean = df.dropna(subset=['Age', 'Income'])

# Drop columns with >40% missing
threshold = 0.4
df_clean = df.loc[:, df.isnull().mean() < threshold]
```

**When safe:** Data is MCAR, few rows are affected (<5% of data).  
**When dangerous:** MNAR or MAR — introduces bias.

### Day 36: Imputing Numerical Data

```python
from sklearn.impute import SimpleImputer
import pandas as pd

df = pd.read_csv('house_prices.csv')
numeric_cols = df.select_dtypes(include='number').columns

# Mean imputation (sensitive to outliers)
imputer_mean = SimpleImputer(strategy='mean')

# Median imputation (robust to outliers — prefer for skewed data)
imputer_median = SimpleImputer(strategy='median')

# Constant imputation
imputer_const = SimpleImputer(strategy='constant', fill_value=0)

# Fit on training, transform both
imputer_median.fit(X_train[numeric_cols])
X_train[numeric_cols] = imputer_median.transform(X_train[numeric_cols])
X_test[numeric_cols]  = imputer_median.transform(X_test[numeric_cols])

print("Mean values learned:", imputer_median.statistics_)
```

### Day 37: Handling Missing Categorical Data

```python
from sklearn.impute import SimpleImputer

# Most frequent (mode) imputation
imputer_mode = SimpleImputer(strategy='most_frequent')

# Constant — add 'Missing' as a new category
imputer_missing = SimpleImputer(strategy='constant', fill_value='Missing')

# Apply to categorical columns
cat_cols = df.select_dtypes(include='object').columns
imputer_mode.fit(X_train[cat_cols])
X_train[cat_cols] = imputer_mode.transform(X_train[cat_cols])
X_test[cat_cols]  = imputer_mode.transform(X_test[cat_cols])
```

### Day 38: Missing Indicator

Adding a binary indicator column marks which rows were imputed — this can be predictive if missingness is informative (MNAR).

```python
from sklearn.impute import SimpleImputer, MissingIndicator
from sklearn.pipeline import FeatureUnion, Pipeline
import numpy as np

# MissingIndicator adds binary columns for each column that had NaN
indicator = MissingIndicator(features='missing-only')
indicators = indicator.fit_transform(X_train)
indicator_df = pd.DataFrame(indicators, columns=[f'{c}_missing' for c in X_train.columns[indicator.features_]])

# Combine with imputed data
X_train_with_indicators = pd.concat([
    pd.DataFrame(imputer_median.transform(X_train), columns=X_train.columns),
    indicator_df
], axis=1)
```

### Day 39: KNN Imputer

Uses K-nearest neighbours to fill missing values — each missing value is replaced by the weighted average of K most similar rows (based on non-missing features).

```python
from sklearn.impute import KNNImputer
import pandas as pd

df = pd.read_csv('pima_diabetes.csv')

# Replace zeros with NaN (zeros mean missing in this dataset)
cols_with_zero = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_zero] = df[cols_with_zero].replace(0, np.nan)

knn_imputer = KNNImputer(n_neighbors=5, weights='uniform')
df_imputed = pd.DataFrame(knn_imputer.fit_transform(df), columns=df.columns)

# Compare distributions before/after
print("Before:", df['Insulin'].describe())
print("After:", df_imputed['Insulin'].describe())
```

### Day 40: IterativeImputer (MICE)

The most sophisticated imputation method. Models each feature with missing values as a function of all other features, iterates until convergence.

```python
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor

# Use Random Forest as the estimator for each feature
mice_imputer = IterativeImputer(
    estimator=RandomForestRegressor(n_estimators=10, random_state=42),
    max_iter=10,
    random_state=42,
    verbose=1
)

df_mice = pd.DataFrame(
    mice_imputer.fit_transform(df_with_missing),
    columns=df_with_missing.columns
)
```

---

## Days 42–44 — Outlier Detection and Removal

### Day 42: Z-Score Method

```python
from scipy import stats
import numpy as np

# Identify outliers: |z-score| > 3
z_scores = np.abs(stats.zscore(df['residual sugar']))
outlier_mask = z_scores > 3
print(f"Outliers found: {outlier_mask.sum()}")

# Remove outliers
df_clean = df[~outlier_mask]

# Multi-column Z-score
z_all = np.abs(stats.zscore(df.select_dtypes(include=np.number)))
df_clean = df[(z_all < 3).all(axis=1)]
```

### Day 43: IQR Method

```python
def remove_outliers_iqr(df, col, multiplier=1.5):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - multiplier * IQR
    upper = Q3 + multiplier * IQR
    return df[(df[col] >= lower) & (df[col] <= upper)]

df_clean = remove_outliers_iqr(df, 'CRIM')
print(f"Removed {len(df) - len(df_clean)} outliers from CRIM")
```

### Day 44: Percentile Capping (Winsorisation)

Rather than removing outliers (which loses data), cap them at the 1st and 99th percentile.

```python
def winsorise(df, col, lower_pct=0.01, upper_pct=0.99):
    lower = df[col].quantile(lower_pct)
    upper = df[col].quantile(upper_pct)
    df[col] = df[col].clip(lower=lower, upper=upper)
    return df

df = winsorise(df, 'price')

# Sklearn equivalent
from sklearn.preprocessing import RobustScaler  # not winsorisation but robust
# or use custom transformer
from sklearn.base import BaseEstimator, TransformerMixin

class Winsoriser(BaseEstimator, TransformerMixin):
    def __init__(self, lower=0.01, upper=0.99):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        self.lower_ = pd.Series(X).quantile(self.lower)
        self.upper_ = pd.Series(X).quantile(self.upper)
        return self

    def transform(self, X, y=None):
        return pd.Series(X).clip(self.lower_, self.upper_).values
```

---

## Day 45 — Feature Construction & Splitting

### Feature Construction (Creating New Features)

```python
df = pd.read_csv('pima_diabetes.csv')

# 1. Interaction features
df['Glucose_BMI'] = df['Glucose'] * df['BMI']
df['Age_Glucose']  = df['Age'] * df['Glucose']

# 2. Ratio features
df['Glucose_per_Insulin'] = df['Glucose'] / (df['Insulin'] + 1e-8)

# 3. Polynomial features
from sklearn.preprocessing import PolynomialFeatures
poly = PolynomialFeatures(degree=2, include_bias=False, interaction_only=False)
X_poly = poly.fit_transform(df[['Glucose', 'BMI']])
# Creates: Glucose, BMI, Glucose^2, Glucose*BMI, BMI^2

# 4. Aggregation features (for grouped data)
df['Family_Size'] = df['SibSp'] + df['Parch'] + 1  # Titanic example
df['Is_Alone']    = (df['Family_Size'] == 1).astype(int)
```

### Feature Splitting

```python
# Split name into title and surname (Titanic)
df['Title'] = df['Name'].str.extract(r', ([A-Za-z]+)\.')
df['Surname'] = df['Name'].str.split(',').str[0]

# Split address into components
df['City']    = df['Location'].str.split(',').str[0].str.strip()
df['Country'] = df['Location'].str.split(',').str[1].str.strip()

# Extract numbers from strings
df['Floor_num'] = df['Floor'].str.extract(r'(\d+)').astype(float)
```

---

## Day 46 — Feature Selection

### Why Select Features?
- Fewer features → simpler model → better generalisation
- Removes noise from irrelevant features
- Reduces training time
- Improves interpretability

### Method 1: Variance Threshold (Remove Near-Zero Variance)

```python
from sklearn.feature_selection import VarianceThreshold

selector = VarianceThreshold(threshold=0.01)  # remove features with var < 0.01
X_selected = selector.fit_transform(X)
print(f"Before: {X.shape[1]}, After: {X_selected.shape[1]}")
```

### Method 2: Univariate Selection (SelectKBest)

```python
from sklearn.feature_selection import SelectKBest, f_classif, chi2, mutual_info_classif

# F-statistic (assumes linear relationship)
selector = SelectKBest(f_classif, k=10)
X_selected = selector.fit_transform(X, y)

# Mutual information (captures non-linear)
selector_mi = SelectKBest(mutual_info_classif, k=10)
X_selected_mi = selector_mi.fit_transform(X, y)

print("Selected features:", X.columns[selector.get_support()].tolist())
```

### Method 3: Recursive Feature Elimination (RFE)

```python
from sklearn.feature_selection import RFE, RFECV
from sklearn.linear_model import LogisticRegression

# RFE — manually specify number of features
rfe = RFE(estimator=LogisticRegression(max_iter=1000), n_features_to_select=10)
rfe.fit(X_train, y_train)
print("Selected features:", X_train.columns[rfe.support_].tolist())
print("Feature rankings:", rfe.ranking_)

# RFECV — automatically finds optimal number via cross-validation
rfecv = RFECV(estimator=LogisticRegression(max_iter=1000), cv=5, scoring='accuracy')
rfecv.fit(X_train, y_train)
print("Optimal number of features:", rfecv.n_features_)
```

### Method 4: Feature Importance from Trees

```python
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

importances = pd.Series(rf.feature_importances_, index=X_train.columns)
importances = importances.sort_values(ascending=False)

plt.figure(figsize=(10, 6))
importances[:15].plot(kind='bar')
plt.title('Top 15 Feature Importances (Random Forest)')
plt.tight_layout()
plt.show()
```

---

## Day 47 — PCA (Principal Component Analysis)

### What is PCA?
PCA is a **dimensionality reduction** technique that projects data onto a new coordinate system where:
- The first axis (PC1) captures maximum variance
- The second axis (PC2) captures maximum remaining variance, orthogonal to PC1
- And so on...

This allows you to keep most of the information with far fewer features.

### Mathematical Intuition
1. Standardise the features (PCA is scale-sensitive)
2. Compute the **covariance matrix** of the standardised features
3. Find the **eigenvalues** (how much variance each component captures) and **eigenvectors** (the direction of each component)
4. Sort by eigenvalue descending — these are your principal components
5. Project data onto the top K components

### Implementation

```python
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import numpy as np

# Load and standardise
X = df.drop('target', axis=1)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit PCA
pca = PCA()
pca.fit(X_scaled)

# Explained variance ratio
explained_var = pca.explained_variance_ratio_
cumulative_var = np.cumsum(explained_var)

# Scree plot — choose elbow point
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.bar(range(1, len(explained_var)+1), explained_var)
plt.xlabel('Principal Component')
plt.ylabel('Explained Variance Ratio')
plt.title('Scree Plot')

plt.subplot(1, 2, 2)
plt.plot(range(1, len(cumulative_var)+1), cumulative_var, 'bo-')
plt.axhline(0.95, color='red', linestyle='--', label='95% threshold')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('Cumulative Variance')
plt.legend()
plt.tight_layout()
plt.show()

# Find components needed for 95% variance
n_components_95 = np.argmax(cumulative_var >= 0.95) + 1
print(f"Components needed for 95% variance: {n_components_95}")

# Apply PCA
pca_final = PCA(n_components=n_components_95)
X_pca = pca_final.fit_transform(X_scaled)
print(f"Original shape: {X_scaled.shape}, Reduced shape: {X_pca.shape}")
```

### Visualising MNIST with PCA

```python
from sklearn.datasets import load_digits

digits = load_digits()
X_digits = digits.data    # 1797 samples, 64 features
y_digits = digits.target

# Reduce to 2D for visualisation
pca_2d = PCA(n_components=2)
X_2d = pca_2d.fit_transform(StandardScaler().fit_transform(X_digits))

plt.figure(figsize=(10, 8))
scatter = plt.scatter(X_2d[:, 0], X_2d[:, 1], c=y_digits, cmap='tab10', alpha=0.6, s=10)
plt.colorbar(scatter, label='Digit')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('MNIST Digits — PCA 2D Projection')
plt.show()
```

### When NOT to Use PCA
- When interpretability matters (PCA components are not interpretable)
- When features are already low-dimensional
- When you need to identify which original features matter (use feature importance instead)
- When data is not approximately linearly structured (use t-SNE or UMAP instead)